In [ ]:
import os

os.makedirs("../outputs/ml", exist_ok=True)
os.makedirs("../models", exist_ok=True)

print("ML output folders ready.")

In [ ]:
import pathlib
import pandas as pd
import sklearn.ensemble
import sklearn.model_selection

# ---------------------------------------------------------
# 1. Automatically Find and Load Dataset
# ---------------------------------------------------------
project_dir = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
csv_files = [
    f for f in project_dir.rglob("*.csv")
    if ".venv" not in f.parts and f.stat().st_size > 0
]

if not csv_files:
    raise FileNotFoundError("No valid CSV file found in project workspace.")

# Load main dataset
data_path = max(csv_files, key=lambda f: f.stat().st_size)
print(f"Loaded dataset: {data_path.resolve()}")
customer_segments = pd.read_csv(data_path)

# ---------------------------------------------------------
# 2. Select Features (X) and Target Column (y)
# ---------------------------------------------------------
# Filter numeric columns only (required for Random Forest)
numeric_df = customer_segments.select_dtypes(include=["number"]).dropna()

# Auto-detect target column (or update 'total_spent' to your target variable name)
possible_targets = ["total_spent", "total_orders", "avg_order_value", "spending"]
target_col = next((c for c in possible_targets if c in numeric_df.columns), numeric_df.columns[-1])

print(f"Target column chosen: '{target_col}'")

X = numeric_df.drop(columns=[target_col])
y = numeric_df[target_col]

# ---------------------------------------------------------
# 3. Train Model and Predict
# ---------------------------------------------------------
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf_model = sklearn.ensemble.RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

# ---------------------------------------------------------
# 4. Build and Save Forecast Comparison
# ---------------------------------------------------------
forecast_comparison = pd.DataFrame(
    {"Actual": y_test.values, "Predicted": y_pred}
)
forecast_comparison["Error"] = forecast_comparison["Actual"] - forecast_comparison["Predicted"]

output_dir = pathlib.Path("../outputs/ml")
output_dir.mkdir(parents=True, exist_ok=True)

save_file = output_dir / "forecast_comparison.csv"
forecast_comparison.to_csv(save_file, index=False)

print(f"Forecast comparison saved to: {save_file.resolve()}")
forecast_comparison.head()

In [ ]:
from pathlib import Path
import pandas as pd

ml_output_path = Path("../outputs/ml")

print("ML OUTPUT FILES")
print("=" * 50)

for file in sorted(ml_output_path.glob("*.csv")):
    print(file.name)

In [ ]:
import pandas as pd
from pathlib import Path

# ML output folder
ml_path = Path("../outputs/ml")

# Load ML outputs
customer_segments = pd.read_csv(
    ml_path / "customer_segments.csv"
)

demand_forecast = pd.read_csv(
    ml_path / "demand_forecast.csv"
)

forecast_comparison = pd.read_csv(
    ml_path / "forecast_comparison.csv"
)

model_comparison = pd.read_csv(
    ml_path / "model_comparison.csv"
)

print("========== ML OUTPUT SUMMARY ==========")

print("\nCustomer Segments:")
print("Shape:", customer_segments.shape)
print("Columns:", customer_segments.columns.tolist())

print("\nDemand Forecast:")
print("Shape:", demand_forecast.shape)
print("Columns:", demand_forecast.columns.tolist())

print("\nForecast Comparison:")
print("Shape:", forecast_comparison.shape)
print("Columns:", forecast_comparison.columns.tolist())

print("\nModel Comparison:")
print("Shape:", model_comparison.shape)
print("Columns:", model_comparison.columns.tolist())

In [ ]:
print("MODEL COMPARISON")
print("=" * 50)

display(
    model_comparison.sort_values(
        by="RMSE",
        ascending=True
    )
)

In [ ]:
print("DEMAND FORECAST")
print("=" * 50)

display(demand_forecast.head(10))

In [ ]:
print("FORECAST COMPARISON")
print("=" * 50)

display(forecast_comparison.head(10))

In [ ]:
print("CUSTOMER SEGMENTATION")
print("=" * 50)

print("Shape:", customer_segments.shape)

display(customer_segments.head())

print("\nSegment distribution:")
display(
    customer_segments["customer_segment"]
    .value_counts()
)